# Domino Detection & Pip Classification — Training Notebook

This notebook trains **two machine learning models** that work together to read dominoes from a photo:

1. **Domino Detector (YOLO26)** — Finds every domino tile in an image and draws a bounding box around it.
2. **Pip Classifier (CNN)** — Takes each detected domino, splits it in half, and counts the pips (dots) on each side (0–15).

### Why Two Models Instead of One?

A single model that recognizes every possible domino tile would need **136 classes** (every combination in a double-15 set). That requires an enormous training dataset and doesn't scale. By splitting the problem in two, we only need **17 total classes** (1 for detection + 16 for pip counting), and the training data stays manageable.

| Approach | Classes | Training Data Needed |
|----------|---------|---------------------|
| Single model (whole tile) | 136 for double-15 | Enormous |
| Two-stage (detect + classify) | 1 + 16 = **17 total** | Manageable |

### How It Works End-to-End

Once both models are trained and exported to the app:

> **Photo → YOLO26 detects each domino → Crop each detection → Split in half → Classify pip count (0–15) → Score calculation**

### What You'll Need

- **Google Colab** with GPU runtime (or any machine with a GPU)
- **~25 photos** of dominoes labeled with bounding boxes via [Label Studio](https://labelstud.io/)
- **Additional photos** for building the pip classification dataset
- Patience for manually sorting cropped domino halves into pip-count folders

---

## Environment Check

Verify you have a GPU assigned before running anything else. If no GPU is detected, go to **Runtime → Change runtime type → GPU**.

In [ ]:
# Uncomment the line below to install dependencies (required for Colab)
# !pip install ultralytics tensorflow pillow matplotlib scikit-learn

In [ ]:
import platform, subprocess, shutil

print(f'Python:  {platform.python_version()}')
print(f'OS:      {platform.system()} {platform.release()}')

# GPU info
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip()
    for line in gpu_info.split('\n'):
        name, mem, driver = [x.strip() for x in line.split(',')]
        print(f'GPU:     {name} ({mem}) — driver {driver}')
except FileNotFoundError:
    print('GPU:     None detected (nvidia-smi not found)')
    print('         ⚠️  Go to Runtime → Change runtime type → select a GPU')

# CUDA
try:
    cuda_ver = subprocess.check_output(['nvcc', '--version'], text=True)
    for line in cuda_ver.split('\n'):
        if 'release' in line:
            print(f'CUDA:    {line.strip().split("release ")[-1]}')
            break
except FileNotFoundError:
    print('CUDA:    not found')

# Disk space
total, used, free = shutil.disk_usage('/')
print(f'Disk:    {free / (1024**3):.1f} GB free of {total / (1024**3):.1f} GB')

# RAM
try:
    with open('/proc/meminfo') as f:
        for line in f:
            if line.startswith('MemTotal'):
                mem_gb = int(line.split()[1]) / (1024**2)
                print(f'RAM:     {mem_gb:.1f} GB')
                break
except FileNotFoundError:
    pass  # not on Linux

In [ ]:
import os
import shutil
import numpy as np
from pathlib import Path

# All paths are relative to the machine_learning/ directory
ML_ROOT = Path('../').resolve()
DATA_DIR = ML_ROOT / 'data'
DETECTOR_DIR = DATA_DIR / 'detector'
CLASSIFIER_DIR = DATA_DIR / 'classifier'
MODELS_DIR = ML_ROOT / 'models'

# Print paths so you can verify they're correct
print(f'ML Root:         {ML_ROOT}')
print(f'Detector data:   {DETECTOR_DIR}')
print(f'Classifier data: {CLASSIFIER_DIR}')
print(f'Models output:   {MODELS_DIR}')

---

## Stage 1: Train the Domino Detector (YOLO26)

The first model learns to find domino tiles in a photo. It doesn't care what's on the domino — it just draws a box around each one.

**Before running this section**, you need labeled data:
1. Take ~25 photos of dominoes (varying backgrounds, lighting, arrangements)
2. Label them in [Label Studio](https://labelstud.io/) with bounding boxes (single class: `domino`)
3. Export in **YOLO format** — this gives you an `images/` and `labels/` folder

See `machine_learning/docs/ml-pipeline.md` for detailed labeling instructions.

### 1.1 — Import & Split the Label Studio Export

This function takes your Label Studio YOLO export and splits it into training (80%) and validation (20%) sets.

In [ ]:
from sklearn.model_selection import train_test_split

def split_label_studio_export(export_dir, output_dir, val_split=0.2):
    """
    Takes a Label Studio YOLO export and splits it into train/val sets.
    
    Args:
        export_dir: Path to the unzipped Label Studio export (contains images/ and labels/)
        output_dir: Where to save the split data (will create train/ and val/ subdirectories)
        val_split: Fraction of data to use for validation (default: 20%)
    """
    export_dir = Path(export_dir)
    output_dir = Path(output_dir)
    
    images_dir = export_dir / 'images'
    labels_dir = export_dir / 'labels'
    
    # Find all labeled images
    image_files = sorted(list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png')))
    print(f'Found {len(image_files)} labeled images')
    
    # Randomly split into train and val
    train_imgs, val_imgs = train_test_split(image_files, test_size=val_split, random_state=42)
    print(f'Train: {len(train_imgs)} images, Val: {len(val_imgs)} images')
    
    # Copy images and their matching label files into the split directories
    for split_name, split_imgs in [('train', train_imgs), ('val', val_imgs)]:
        img_out = output_dir / split_name / 'images'
        lbl_out = output_dir / split_name / 'labels'
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        
        for img_path in split_imgs:
            shutil.copy2(img_path, img_out / img_path.name)
            label_path = labels_dir / f'{img_path.stem}.txt'
            if label_path.exists():
                shutil.copy2(label_path, lbl_out / label_path.name)
    
    print('Done! Data is ready for YOLO training.')

# ⬇️ UPDATE THIS PATH to point to your Label Studio YOLO export folder
# split_label_studio_export('path/to/label-studio-export', DETECTOR_DIR)

### 1.2 — Create the YOLO Data Config

YOLO needs a `data.yaml` file that tells it where the images are and what classes to detect. We only have one class: `domino`.

In [ ]:
data_yaml_content = f"""path: {DETECTOR_DIR}
train: train/images
val: val/images

nc: 1
names: ['domino']
"""

data_yaml_path = DETECTOR_DIR / 'data.yaml'
data_yaml_path.parent.mkdir(parents=True, exist_ok=True)
data_yaml_path.write_text(data_yaml_content)
print(f'Wrote config to: {data_yaml_path}\n')
print(data_yaml_content)

### 1.3 — Train the Detector

This loads a pretrained YOLO26-nano model and fine-tunes it on our domino dataset. The nano variant is small enough to run on mobile while still being accurate.

Training takes **~15–30 minutes** on a Colab GPU depending on dataset size.

In [ ]:
from ultralytics import YOLO

# Start from the pretrained YOLO26-nano model
model = YOLO('yolo26n.pt')

# Fine-tune on our domino detection dataset
results = model.train(
    data=str(DETECTOR_DIR / 'data.yaml'),
    epochs=100,
    imgsz=640,       # Input image size
    batch=16,        # Batch size (lower this if you run out of GPU memory)
    project=str(MODELS_DIR),
    name='domino_detector'
)

# The best model weights are saved to:
# models/domino_detector/weights/best.pt

### 1.4 — Validate the Detector

Check how well the model performs on the validation set. **mAP50** (mean Average Precision at 50% IoU) is the key metric — aim for **0.90+** before moving on.

In [ ]:
# Load the best checkpoint and run validation
best_model = YOLO(str(MODELS_DIR / 'domino_detector' / 'weights' / 'best.pt'))
metrics = best_model.val()

print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print()
print('If mAP50 is below 0.90, consider adding more labeled training images.')

---

## Stage 2: Train the Pip Classifier

Now that we have a working domino detector, we use it to build training data for the second model. The pip classifier looks at **one half** of a domino and predicts how many pips (dots) are on it — a number from **0 to 15**.

**The workflow:**
1. Take more photos of dominoes (the more variety, the better)
2. Run the YOLO detector to automatically crop out each domino
3. Split each crop in half (top/bottom or left/right depending on orientation)
4. Manually sort the halves into folders by pip count (0–15)
5. Train a classifier on the sorted images

### 2.1 — Auto-Crop Domino Halves

This function runs the trained detector on a folder of photos, crops each detected domino, and splits the crop in half. The halves are saved to an `unsorted/` folder for you to manually label.

In [ ]:
from PIL import Image

def crop_and_split_detections(model, source_dir, output_dir, conf=0.5):
    """
    Runs YOLO detection on photos, crops each domino, and splits it in half.
    
    The halves are saved to output_dir as individual images. You'll need to
    manually sort them into folders 0/ through 15/ by pip count.
    
    Args:
        model: Trained YOLO model
        source_dir: Folder of domino photos to process
        output_dir: Where to save the cropped halves
        conf: Minimum confidence threshold for detections (default: 0.5)
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    results = model.predict(source=str(source_dir), conf=conf)
    
    count = 0
    for result in results:
        img = Image.open(result.path)
        img_name = Path(result.path).stem
        
        for i, box in enumerate(result.boxes.xyxy):
            x1, y1, x2, y2 = map(int, box)
            crop = img.crop((x1, y1, x2, y2))
            w, h = crop.size
            
            if h > w:
                # Vertical domino — split into top half and bottom half
                half_a = crop.crop((0, 0, w, h // 2))
                half_b = crop.crop((0, h // 2, w, h))
            else:
                # Horizontal domino — split into left half and right half
                half_a = crop.crop((0, 0, w // 2, h))
                half_b = crop.crop((w // 2, 0, w, h))
            
            half_a.save(output_dir / f'{img_name}_d{i}_a.jpg')
            half_b.save(output_dir / f'{img_name}_d{i}_b.jpg')
            count += 1
    
    print(f'Detected {count} dominoes → {count * 2} half-images saved to {output_dir}')
    print()
    print('NEXT STEP: Manually sort these images into folders named 0/ through 15/')
    print('           based on the number of pips visible on each half.')

# ⬇️ UPDATE the source path to a folder of new domino photos
# detector = YOLO(str(MODELS_DIR / 'domino_detector' / 'weights' / 'best.pt'))
# crop_and_split_detections(detector, 'path/to/new-photos', CLASSIFIER_DIR / 'unsorted')

### 2.2 — Prepare the Classification Dataset

After manually sorting the halves from `unsorted/` into a `sorted/` folder with subfolders `0/` through `15/`, run the cells below to:

1. Create the required folder structure
2. Split the sorted images into training (80%) and validation (20%) sets

**Expected folder structure after sorting:**
```
classifier_data/
  sorted/
    0/   ← blank halves
    1/   ← halves with 1 pip
    ...
    15/  ← halves with 15 pips
```

In [ ]:
# Create the train/ and val/ folder structure with subfolders 0–15
for split in ['train', 'val']:
    for pip_count in range(16):
        folder = CLASSIFIER_DIR / split / str(pip_count)
        folder.mkdir(parents=True, exist_ok=True)

print('Created classifier folders:')
print('  train/0, train/1, ... train/15')
print('  val/0,   val/1,   ... val/15')

In [ ]:
def split_classifier_data(sorted_dir, train_dir, val_dir, val_split=0.2):
    """
    Splits manually sorted pip images into train/val sets.
    
    Args:
        sorted_dir: Folder containing subfolders 0/ through 15/ with sorted images
        train_dir: Output training directory
        val_dir: Output validation directory
        val_split: Fraction to hold out for validation (default: 20%)
    """
    sorted_dir = Path(sorted_dir)
    train_dir = Path(train_dir)
    val_dir = Path(val_dir)
    
    total_train = 0
    total_val = 0
    
    for pip_folder in sorted(sorted_dir.iterdir()):
        if not pip_folder.is_dir():
            continue
        
        images = list(pip_folder.glob('*.jpg')) + list(pip_folder.glob('*.png'))
        if not images:
            print(f'  Pip {pip_folder.name}: no images found (skipping)')
            continue
        
        train_imgs, val_imgs = train_test_split(images, test_size=val_split, random_state=42)
        
        for img in train_imgs:
            shutil.copy2(img, train_dir / pip_folder.name / img.name)
        for img in val_imgs:
            shutil.copy2(img, val_dir / pip_folder.name / img.name)
        
        total_train += len(train_imgs)
        total_val += len(val_imgs)
        print(f'  Pip {pip_folder.name}: {len(train_imgs)} train, {len(val_imgs)} val')
    
    print(f'\nTotal: {total_train} training images, {total_val} validation images')

# ⬇️ Run this after you've sorted the unsorted/ halves into sorted/0 through sorted/15
# split_classifier_data(
#     CLASSIFIER_DIR / 'sorted',
#     CLASSIFIER_DIR / 'train',
#     CLASSIFIER_DIR / 'val'
# )

### 2.3 — Train the Pip Classifier

This trains a simple CNN (Convolutional Neural Network) to classify domino halves by pip count. The architecture is intentionally straightforward — it's a baseline that should work well for this task.

> **Note:** The model architecture is a starting point. Once we have real training data, we can evaluate whether a different approach (e.g., transfer learning, MobileNet) performs better.

In [ ]:
import tensorflow as tf

IMG_SIZE = (100, 100)   # All domino halves are resized to this
BATCH_SIZE = 32
NUM_CLASSES = 16        # Pip counts 0 through 15

# Load training and validation images directly from the folder structure
# TensorFlow reads the folder names (0, 1, 2, ..., 15) as class labels
train_ds = tf.keras.utils.image_dataset_from_directory(
    str(CLASSIFIER_DIR / 'train'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(CLASSIFIER_DIR / 'val'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Verify the classes are in the right order (should be 0, 1, 2, ..., 15)
print('Classes:', train_ds.class_names)

In [ ]:
# Define the classifier model
# Architecture: 4 convolutional blocks → flatten → dense layers → 16-class output
model = tf.keras.Sequential([
    # Normalize pixel values from [0, 255] to [0, 1]
    tf.keras.layers.Rescaling(1./255, input_shape=(100, 100, 3)),
    
    # Block 1: Learn basic features (edges, simple shapes)
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    # Block 2: Learn pip-like patterns
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    # Block 3: Learn pip arrangements
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    # Block 4: Learn higher-level combinations
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    # Classification head
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.3),         # Helps prevent overfitting
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Train the model, saving the best checkpoint based on validation accuracy
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    str(MODELS_DIR / 'pip_classifier_best.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[checkpoint]
)

print()
print(f'Best val accuracy: {max(history.history["val_accuracy"]):.4f}')

### 2.4 — Training Curves

Plot accuracy and loss over time. Look for:
- **Training and validation curves tracking together** = good generalization
- **Val accuracy plateauing while train keeps climbing** = overfitting (add more data or increase dropout)
- **Both curves still improving at the end** = try more epochs

---

## Export Models for Mobile

Both models need to be converted to **TFLite** format for the Flutter app. TFLite models are optimized for mobile — they're smaller and run efficiently on-device without needing a server.

### Final output files:
| File | Purpose | Expected Size |
|------|---------|---------------|
| `domino_detector.tflite` | YOLO26 — finds domino tiles in a photo | ~3–4 MB |
| `pip_classifier.tflite` | CNN — classifies pip count per half | ~1 MB |
| `labels_classifier.txt` | Class labels for the classifier (0–15) | tiny |

In [ ]:
# Export the YOLO detector to TFLite (quantized for smaller size and faster inference)
best_yolo = YOLO(str(MODELS_DIR / 'domino_detector' / 'weights' / 'best.pt'))
best_yolo.export(format='tflite', int8=True, imgsz=640)
print('YOLO detector exported to TFLite')

In [ ]:
# Export the pip classifier to TFLite
classifier = tf.keras.models.load_model(str(MODELS_DIR / 'pip_classifier_best.h5'))

converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Quantize for smaller size
tflite_model = converter.convert()

output_path = MODELS_DIR / 'pip_classifier.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f'Pip classifier exported to: {output_path}')
print(f'Size: {len(tflite_model) / 1024:.1f} KB')

In [ ]:
# Generate the labels file (maps class index to pip count)
labels_path = MODELS_DIR / 'labels_classifier.txt'
labels_path.write_text('\n'.join(str(i) for i in range(16)))
print(f'Labels file written to: {labels_path}')
print()
print('All done! Copy these files into your Flutter app\'s assets/models/ directory:')
print('  - domino_detector.tflite')
print('  - pip_classifier.tflite')
print('  - labels_classifier.txt')

In [ ]:
# Generate labels file
labels_path = MODELS_DIR / 'labels_classifier.txt'
labels_path.write_text('\n'.join(str(i) for i in range(16)))
print(f'Labels written to {labels_path}')